In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import collections
import functools
import os
import pathlib
import typing

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

In [4]:
def sanitize_name(name):
    return name.replace(' ', '_').replace('*', '').replace(';', '_and').replace('/', '_')

# Save coarse pathogen tasks

After our MOFA analysis where we see that we do not detect large specific signature for our narrow pathogen groups, we asked what is the signal for coarse pathogen groups:
- Early SARS-CoV-2
- Late SARS-CoV-2
- Pure bacteria

vs Healthy control and NPC, and vs each other.

Here we exclude pathogen-negative samples and mixed bacterial viral samples.

In [ ]:
BASE = common_data.DATA / '05_pseudobulk/30a_pathogens_coarse'

In [6]:
os.makedirs(BASE, exist_ok=True)

In [7]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [8]:
sc_labels.perturbation_groups_2.unique()

array(['discard', 'Gram-*', 'Late SARS-CoV-2; Gram+',
       'Early SARS-CoV-2; Gram+', 'Early SARS-CoV-2', 'Late SARS-CoV-2',
       'NPC', 'Pseudomonas aeruginosa; SARS-CoV-2', 'Gram+',
       'Gram-*; Gram+', 'Pseudomonas aeruginosa', 'Healthy'], dtype=object)

In [9]:
sc_labels['pathogens_coarse'] = sc_labels.perturbation_groups_2.replace({
    'Gram-*': 'Bacteria',
    'Gram+': 'Bacteria',
    'Pseudomonas aeruginosa': 'Bacteria',
    'Gram-*; Gram+': 'Bacteria',
    'Late SARS-CoV-2; Gram+': 'Mixed',
    'Early SARS-CoV-2; Gram+': 'Mixed',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'Mixed',
})

In [10]:
sc_labels.pathogens_coarse.value_counts()

discard             115
Bacteria             58
Early SARS-CoV-2     35
Mixed                33
NPC                  26
Late SARS-CoV-2      25
Healthy               9
Name: pathogens_coarse, dtype: int64

In [11]:
SHORTCUTS = {
    'Early SARS-CoV-2': 'eCOVID',
    'Late SARS-CoV-2': 'lCOVID',
    'Bacteria': 'Bac',
    'Mixed': 'Mix',
    'Healthy': 'H',
    'NPC': 'N'
}

In [12]:
# manually define
tasks = [
    ('Healthy', 'Early SARS-CoV-2'),
    ('Healthy', 'Late SARS-CoV-2'),
    ('Healthy', 'Bacteria'),
    ('Healthy', 'Mixed'),
    ('Healthy', 'NPC'),
    ('NPC', 'Early SARS-CoV-2'),
    ('NPC', 'Late SARS-CoV-2'),
    ('NPC', 'Bacteria'),
    ('NPC', 'Mixed'),
    ('Early SARS-CoV-2', 'Late SARS-CoV-2'),
    ('Early SARS-CoV-2', 'Bacteria'),
    ('Early SARS-CoV-2', 'Mixed'),
    ('Late SARS-CoV-2', 'Bacteria'),
    ('Late SARS-CoV-2', 'Mixed'),
    ('Bacteria', 'Mixed')
]

In [13]:
def get_task_sample_groups(
    info: common_data.TaskInfo,
    labels: pd.DataFrame
) -> typing.Dict[str, typing.Collection[str]]:
    col = info.column
    sample_data = labels.loc[labels[col].isin(info.column_values)].groupby(col).bal_barcode
    return dict(tuple(sample_data))

In [14]:
def save_task(
    task: common_data.TaskInfo,
):
    path = BASE / task.pathname
    metas = []
    for group, samples in task.samples.items():
        metas.append(pd.DataFrame({
            'bal_barcode': list(samples),
            'group': group
        }))
    os.makedirs(path, exist_ok=True)
    meta = pd.concat(metas, axis=0)
    meta.to_csv(path / '_group_labels.csv')

In [15]:
infos = []
for task in tasks:
    info = common_data.TaskInfo(
        pathname=f'{SHORTCUTS[task[0]]}_vs_{SHORTCUTS[task[1]]}',
        column='pathogens_coarse',
        column_values=task,
        split_column=None
    )
    info.samples = get_task_sample_groups(info, sc_labels)
    infos.append(info)

In [16]:
for task in infos:
    if len(task.samples) < 2:
        continue
    save_task(task)